In [1]:
import os
import sys
import torch
import yaml
import pandas as pd
import numpy as np
import cProfile
import pstats

from itertools import count
from glob import glob
from tqdm import tqdm

import flow_package as fp
from flow_package.multi_df_env import MultiDfEnv, EnvConfig

from utils import setup_logging, rolling_normalize
from network import DeepFlowNetwork
from network_v2 import DeepFlowNetworkV2

# 追加インポート: mlflow とプロットユーティリティ
import mlflow
import plot as plot_lib
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

os.environ["MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING"] = "true"



In [2]:
# 追加: デバイス設定を学習コードと同じロジックで統一
if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")


def load_params():
    if len(sys.argv) != 2:
        print("Usage: python src/evaluate_drl.py <input_file_directory>")
        sys.exit(1)

    input_path = "../data/test/binary"
    all_params = yaml.safe_load(open("../params.yaml"))

    # setup_mlflow(all_params)
    # params = all_params["evaluate_drl"]
    return all_params, input_path


def setup_mlflow(all_params):
    if all_params["mlflow"]["use_azure"]:
        path = os.path.join(os.path.dirname(__file__), "..", "config.json")
        print(path)
        ml_client = MLClient.from_config(
            credential=DefaultAzureCredential(),
            config_path=path
        )
        mlflow_tracking_uri = ml_client.workspaces.get(ml_client.workspace_name).mlflow_tracking_uri
    else:
        mlflow_tracking_uri = all_params["mlflow"]["tracking_uri"]
    if all_params["mlflow"]["use_dagshub"]:
        import dagshub
        dagshub.init(repo_owner='liverHawk', repo_name='research_data_drl', mlflow=True)
    
    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(
        f"{all_params['mlflow']['experiment_name']}_evaluate_drl"
    )


def load_csv(input):
    files = glob(os.path.join(input, "*.csv.gz"))
    df = pd.concat([
        pd.read_csv(f) for f in files
    ])
    df = df.reset_index(drop=True)
    df = rolling_normalize(df)
    return df


def write_result(cm_memory, prefix):
    # cm_memory: list of [predicted, actual]
    cm_data = []
    for pred, actual in cm_memory:
        pred_val = pred.item() if hasattr(pred, "item") else int(pred)
        actual_val = actual.item() if hasattr(actual, "item") else int(actual)
        cm_data.append([pred_val, actual_val])

    if len(cm_data) == 0:
        # 空の場合は空のファイルを作るだけ
        os.makedirs("evaluate", exist_ok=True)
        cm_path = os.path.join("evaluate", f"{prefix}_confusion_matrix.csv")
        pd.DataFrame(columns=["Predicted", "Actual"]).to_csv(cm_path, index=False)
        return None, 0.0, cm_path

    # クラス数はデータ上の最大ラベルから推定
    preds = [p for p, a in cm_data]
    actuals = [a for p, a in cm_data]
    n = max(max(preds), max(actuals)) + 1
    cm = np.zeros((n, n), dtype=int)

    # plot.py のラベル付け（x: Actual, y: Predicted）に合わせて cm[predicted][actual] を増やす
    for pred, actual in cm_data:
        cm[pred][actual] += 1

    os.makedirs("evaluate", exist_ok=True)
    cm_path = os.path.join("evaluate", f"{prefix}_confusion_matrix.csv")
    # CSV 保存（行: Predicted, 列: Actual のマトリクス形式）
    pd.DataFrame(cm).to_csv(cm_path, index=True)

    total = cm.sum()
    accuracy = float(np.trace(cm) / total) if total > 0 else 0.0
    return cm, accuracy, cm_path


def to_tensor(state, include_category=True):
    # 学習側と同じように device を渡す
    if include_category:
        return fp.to_tensor(state, device=device)
    else:
        return torch.tensor(state, device=device, dtype=torch.float32)

In [3]:
all_params, input_path = load_params()
df = load_csv(input_path)

In [4]:
drl_options = all_params.get("drl_options", {})
params = all_params.get("evaluate_drl", {})

input = EnvConfig(
    data=df,
    label_column="Label",
    render_mode=None,
    max_steps=drl_options.get("max_steps", 100),
    normalize_method="minmax",
    rolling_window=drl_options.get("rolling_window", 10),
    test_mode=True,
)
env = MultiDfEnv(input)

print(params)
include_category = params.get("include_category", True)

# 環境から正しい次元を取得してモデルを同じ呼び出し方で作る
n_states = env.observation_space.shape[0]
n_actions = env.action_space.n

if include_category:
    network = DeepFlowNetwork(n_states, n_actions).to(device)
else:
    network = DeepFlowNetworkV2(n_states, n_actions).to(device)

# 学習側で保存したファイル名に合わせて読み込む
if include_category:
    path = os.path.join("..", "model", "drl_model_with_category.pth")
else:
    path = os.path.join("..", "model", "drl_model.pth")

# デバイスを考慮してロード
network.load_state_dict(torch.load(path, map_location=device))
network.eval()

cm_memory = []

log_path = os.path.join("evaluate_drl.log")
logger = setup_logging(log_path)
logger.info("Starting evaluation...")

sum_rewards = 0.0

def select_action(states_tensor):
    with torch.no_grad():
        p_actions = network(states_tensor)
        

# for i_loop in range(1):
raw_state, _ = env.reset()

states_tensor = []
for _ in range(10):
    state, _, _, _, _ = env.step(0)  # ダミーアクションで状態を収集
    states_tensor.append(to_tensor(state, include_category=include_category))

states_tensor = torch.stack(states_tensor).unsqueeze(0)  # バッチ次元を追加
p_actions = network(states_tensor)

# try:
#     state = to_tensor(raw_state, include_category=include_category)
# except Exception as e:
#     raise ValueError(f"Error converting state to tensor: {e}")

# for t in count():
#     with torch.no_grad():
#         predicted_action = network(state)
#         if predicted_action.dim() == 1:
#             predicted_action = predicted_action.unsqueeze(0)
#         predicted_action = predicted_action.max(1)[1].view(1, 1)

#     raw_next_state, reward, terminated, _, info = env.step(predicted_action.item())
#     sum_rewards += reward
#     # predicted と actual を混同行列用に保存（predicted, actual）
#     actual_answer = int(info["confusion_matrix_index"][1])
#     cm_memory.append([predicted_action, actual_answer])

#     if terminated:
#         break
#     try:
#         next_state = to_tensor(raw_next_state, include_category=include_category)
#     except Exception as e:
#         raise ValueError(f"Error converting next state to tensor: {e}")

#     state = next_state
#     progress_bar.update(1)

# logger.info("Evaluation completed.")


2025-10-18 16:28:38 TTM1Pro.local my_logger[81491] INFO Starting evaluation...


{'include_category': False, 'test': 10}


In [5]:
print(p_actions)
print(p_actions[0, 0])
# print(p_actions[0, :].max(1))
print(torch.argmax(p_actions[0, :], dim=1))
print(torch.argmax(p_actions[0, :], dim=1).unsqueeze(0))
print(torch.argmax(p_actions[0, :], dim=1).unsqueeze(1))
# predicted_action = torch.argmax(p_actions[0, 0], dim=1)
# print(predicted_action)

tensor([[[ 0.1592, -2.2366, -1.9879, -1.7820, -2.1304, -1.7397, -2.6478,
          -1.3873, -1.9737, -2.0327],
         [ 0.9052, -2.0517, -1.7649, -1.4457, -2.2847, -1.2668, -2.5698,
          -1.4716, -1.3018, -1.9188],
         [ 0.1544, -2.2725, -1.9319, -1.8209, -2.1852, -1.8453, -2.4384,
          -1.4982, -1.9966, -2.0495],
         [ 0.3055, -2.1149, -1.9402, -1.6147, -2.1061, -1.3726, -2.3653,
          -1.2496, -1.6761, -1.9450],
         [ 0.8272, -2.1100, -1.8062, -1.6897, -2.3444, -1.6248, -2.2508,
          -1.4484, -1.4694, -1.2067],
         [ 0.6850, -2.0450, -1.7820, -1.6154, -2.1888, -1.5243, -2.3693,
          -1.5035, -1.4508, -1.6685],
         [ 0.3380, -1.7834, -1.4875, -1.5363, -1.4821, -1.3458, -1.6798,
          -0.6210, -1.6923, -1.1572],
         [ 0.5726, -2.1225, -1.6817, -1.9290, -2.0497, -1.8980, -1.6126,
          -1.7987, -1.9044, -1.5838],
         [ 0.5251, -1.8620, -1.3717, -1.6041, -1.5697, -1.3404, -1.7141,
          -0.8285, -1.5120, -1.2517],
 

In [6]:
states, _, _, _, _ = env.step(torch.argmax(p_actions[0, :], dim=1))

RuntimeError: Boolean value of Tensor with more than one value is ambiguous